# Prefix-Conditioned Suffix Scoring（Revised, Teacher-Forced）

本版本针对“信号太弱”做了三项最小改进（仍不改仓库源码）：
1. 使用更强 corruption 候选，并自动选择 clean->corrupted drop 最大的方案。
2. 保持 target suffix 不变，只破坏 prefix 条件。
3. 对多个 patch site 做小范围 sweep，选择 recovery gain 最强的 site。

checkpoint 固定路径：
- /home/cody/LLM_ML_CODE_DATA/FINAL_report_project/toygpt2_runs/tinystories_dual/standard/ckpt_standard_last.pt
- /home/cody/LLM_ML_CODE_DATA/FINAL_report_project/toygpt2_runs/tinystories_dual/attnres/ckpt_attnres_last.pt


In [ ]:
# Cell 1. 环境与导入
import json
import math
import sys
from copy import deepcopy
from pathlib import Path

import torch
import matplotlib.pyplot as plt

CANDIDATES = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = None
for p in CANDIDATES:
    if (p / 'toygpt2').exists():
        REPO_ROOT = p
        break
if REPO_ROOT is None:
    raise RuntimeError('未找到仓库根目录（应包含 toygpt2 目录）')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

NOTEBOOK_NAME = 'prefix_suffix_scoring_revised'
OUTPUT_DIR = REPO_ROOT / 'output' / NOTEBOOK_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def output_path(filename: str) -> Path:
    path = OUTPUT_DIR / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


def _to_serializable(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): _to_serializable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_to_serializable(v) for v in value]
    if torch.is_tensor(value):
        return value.detach().cpu().tolist()
    if hasattr(value, 'item'):
        try:
            return value.item()
        except Exception:
            pass
    return value


def save_json_artifact(payload, filename: str) -> Path:
    path = output_path(filename)
    path.write_text(json.dumps(_to_serializable(payload), ensure_ascii=False, indent=2), encoding='utf-8')
    print('[saved]', path)
    return path


def save_figure_artifact(filename: str, fig=None, *, dpi: int = 180) -> Path:
    fig = plt.gcf() if fig is None else fig
    path = output_path(filename)
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    print('[saved]', path)
    return path

from scripts.evaluate import load_checkpoint
from data.data import build_dataloaders
from interp.patching import patch_from_cache

print('REPO_ROOT:', REPO_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__)


In [ ]:
# Cell 2. 加载 checkpoint 和模型（双模型一起加载）
device = torch.device('cpu')
CHECKPOINTS = {
    'standard': str(REPO_ROOT / 'toygpt2_runs' / 'tinystories_dual' / 'standard' / 'ckpt_standard_last.pt'),
    'attnres': str(REPO_ROOT / 'toygpt2_runs' / 'tinystories_dual' / 'attnres' / 'ckpt_attnres_last.pt'),
}
MODEL_ORDER = list(CHECKPOINTS.keys())

MODELS = {}
EXPERIMENTS = {}
CHECKPOINT_PAYLOADS = {}
MODEL_SUMMARIES = []
for model_key in MODEL_ORDER:
    ckpt_path = Path(CHECKPOINTS[model_key])
    model, experiment, checkpoint = load_checkpoint(ckpt_path, device=device)
    model = model.to(device).eval()
    MODELS[model_key] = model
    EXPERIMENTS[model_key] = experiment
    CHECKPOINT_PAYLOADS[model_key] = checkpoint
    MODEL_SUMMARIES.append(
        {
            'model_variant': model_key,
            'checkpoint': ckpt_path,
            'model_type': checkpoint.get('model_type'),
            'step': checkpoint.get('step'),
            'n_layer': int(experiment.model.n_layer),
            'n_head': int(experiment.model.n_head),
            'n_embd': int(experiment.model.n_embd),
            'vocab_size': int(experiment.model.vocab_size),
            'block_size': int(experiment.model.block_size),
            'device': str(next(model.parameters()).device),
        }
    )

shared_vocab_sizes = {row['model_variant']: row['vocab_size'] for row in MODEL_SUMMARIES}
if len(set(shared_vocab_sizes.values())) != 1:
    raise ValueError(f'两个模型 vocab_size 不一致: {shared_vocab_sizes}')

SHARED_VOCAB_SIZE = next(iter(shared_vocab_sizes.values()))
SHARED_BLOCK_SIZE = min(row['block_size'] for row in MODEL_SUMMARIES)
REFERENCE_MODEL_KEY = MODEL_ORDER[0]

print('loaded model summaries:')
print(json.dumps(MODEL_SUMMARIES, ensure_ascii=False, indent=2))
print('shared_vocab_size:', SHARED_VOCAB_SIZE)
print('shared_block_size:', SHARED_BLOCK_SIZE)
print('reference_model_key:', REFERENCE_MODEL_KEY)

save_json_artifact(
    {
        'device': str(device),
        'model_order': MODEL_ORDER,
        'model_summaries': MODEL_SUMMARIES,
        'shared_vocab_size': SHARED_VOCAB_SIZE,
        'shared_block_size': SHARED_BLOCK_SIZE,
        'reference_model_key': REFERENCE_MODEL_KEY,
    },
    'checkpoint_model_summaries.json',
)


In [ ]:
# Cell 3. 从 TinyStories 抽样（与两个 checkpoint 的公共配置对齐）
reference_experiment = EXPERIMENTS[REFERENCE_MODEL_KEY]
sample_model_config = deepcopy(reference_experiment.model)
sample_model_config.block_size = int(min(48, SHARED_BLOCK_SIZE))

for model_key in MODEL_ORDER:
    model_block_size = int(EXPERIMENTS[model_key].model.block_size)
    if model_block_size < sample_model_config.block_size:
        raise ValueError(f'{model_key} 的 block_size={model_block_size} 小于共享 sample block_size={sample_model_config.block_size}')


tiny_cfg = deepcopy(reference_experiment.data)
tiny_cfg.dataset_type = 'tinystories'
tiny_cfg.block_stride = max(1, sample_model_config.block_size)
if tiny_cfg.train_texts is None:
    tiny_cfg.train_texts = 1024
if tiny_cfg.val_texts is None:
    tiny_cfg.val_texts = 256

train_loader, _ = build_dataloaders(
    model_config=sample_model_config,
    data_config=tiny_cfg,
    batch_size=8,
    num_workers=0,
    seed=2026,
    distributed=False,
    verbose=True,
)
inputs, targets = next(iter(train_loader))
source = 'tinystories_train_loader'

# 重构 full sequence: z = [prefix || suffix], z 长度 = block_size + 1
full_clean = torch.cat([inputs, targets[:, -1:].clone()], dim=1).to(device=device, dtype=torch.long)

full_len = full_clean.size(1)
prefix_len = max(4, int(full_len * 0.50))
if prefix_len >= full_len:
    prefix_len = full_len - 1
suffix_len = full_len - prefix_len
suffix_start = prefix_len

print('sample_source:', source)
print('reference_model_key:', REFERENCE_MODEL_KEY)
print('batch_size:', full_clean.size(0))
print('full_len:', full_len, 'prefix_len:', prefix_len, 'suffix_len:', suffix_len)
print('suffix_start_index(in z):', suffix_start)
print('example clean prefix tokens:', full_clean[0, :prefix_len].tolist())
print('example clean suffix tokens:', full_clean[0, prefix_len:].tolist())


In [ ]:
# Cell 4. 构造更强 corruption（保持 target suffix 不变）
def _mutate_values(orig: torch.Tensor, vocab_size: int, delta: int = 17) -> torch.Tensor:
    # 保证替换后不等于原值
    return (orig + delta) % max(2, vocab_size)


def build_corruption_candidates(full_tokens: torch.Tensor, prefix_len: int, vocab_size: int):
    cands = {}
    meta = {}

    # 候选1：边界关键位点 + 邻域多点扰动
    c1 = full_tokens.clone()
    pos1 = sorted(set([max(1, prefix_len - 1), max(1, prefix_len - 2), max(1, prefix_len - 3)]))
    old1 = c1[:, pos1].clone()
    c1[:, pos1] = _mutate_values(c1[:, pos1], vocab_size, delta=11)
    cands['boundary_window_flip'] = c1
    meta['boundary_window_flip'] = {'positions': pos1, 'old_sample0': old1[0].tolist(), 'new_sample0': c1[0, pos1].tolist()}

    # 候选2：prefix 后半段随机化（更强）
    c2 = full_tokens.clone()
    start2 = max(1, prefix_len // 2)
    pos2 = list(range(start2, prefix_len))
    if pos2:
        torch.manual_seed(123)
        rand = torch.randint(0, vocab_size, (c2.size(0), len(pos2)), device=c2.device)
        # 避免恰好等于原值
        eq = rand == c2[:, pos2]
        rand = torch.where(eq, _mutate_values(rand, vocab_size, delta=7), rand)
        old2 = c2[:, pos2].clone()
        c2[:, pos2] = rand
        cands['prefix_tail_randomize'] = c2
        meta['prefix_tail_randomize'] = {
            'positions': [pos2[0], pos2[-1], len(pos2)],
            'old_sample0_head': old2[0, :min(6, len(pos2))].tolist(),
            'new_sample0_head': c2[0, pos2][:min(6, len(pos2))].tolist(),
        }

    # 候选3：prefix 后半段逆序（结构性破坏）
    c3 = full_tokens.clone()
    start3 = max(1, prefix_len // 2)
    pos3 = list(range(start3, prefix_len))
    if len(pos3) >= 2:
        old3 = c3[:, pos3].clone()
        c3[:, pos3] = torch.flip(c3[:, pos3], dims=[1])
        cands['prefix_tail_reverse'] = c3
        meta['prefix_tail_reverse'] = {
            'positions': [pos3[0], pos3[-1], len(pos3)],
            'old_sample0_head': old3[0, :min(6, len(pos3))].tolist(),
            'new_sample0_head': c3[0, pos3][:min(6, len(pos3))].tolist(),
        }

    # 强约束：suffix 保持不变
    for name, cand in cands.items():
        assert torch.equal(cand[:, prefix_len:], full_tokens[:, prefix_len:]), f'{name} 改动了 suffix，非法。'

    return cands, meta


corruption_candidates, corruption_meta = build_corruption_candidates(
    full_tokens=full_clean,
    prefix_len=prefix_len,
    vocab_size=SHARED_VOCAB_SIZE,
)

print('corruption candidate names:', list(corruption_candidates.keys()))
for name in corruption_candidates:
    info = corruption_meta[name]
    print(f"\n[{name}] info:")
    print(json.dumps(info, ensure_ascii=False, indent=2))
    diff_positions = (corruption_candidates[name][0, :prefix_len] != full_clean[0, :prefix_len]).nonzero(as_tuple=False).view(-1).tolist()
    print(f"[{name}] sample0 changed positions in prefix:", diff_positions)

print('suffix unchanged check:', all(torch.equal(v[:, prefix_len:], full_clean[:, prefix_len:]) for v in corruption_candidates.values()))

save_json_artifact(corruption_meta, 'corruption_candidates.json')


In [ ]:
# Cell 5. scoring helper（双模型 notebook glue code）
def scalar_metrics_view(metrics: dict):
    return {
        'suffix_sum_logprob': float(metrics['suffix_sum_logprob']),
        'suffix_avg_logprob': float(metrics['suffix_avg_logprob']),
        'exact_hit': float(metrics['exact_hit']),
        'topk_hit': float(metrics['topk_hit']),
        'avg_rank': float(metrics['avg_rank']),
    }


def suffix_metrics_from_logits(logits: torch.Tensor, target_full_tokens: torch.Tensor, prefix_len: int, topk: int = 5):
    # logits: [B, L-1, V], target_full_tokens: [B, L]
    targets = target_full_tokens[:, 1:]  # [B, L-1]
    start = prefix_len - 1
    if start < 0 or start >= targets.size(1):
        raise ValueError(f'非法 prefix_len={prefix_len} for targets_len={targets.size(1)}')

    suffix_targets = targets[:, start:]   # [B, S]
    suffix_logits = logits[:, start:, :]  # [B, S, V]

    log_probs = torch.log_softmax(suffix_logits, dim=-1)
    per_pos_logprob = torch.gather(log_probs, -1, suffix_targets.unsqueeze(-1)).squeeze(-1)  # [B, S]

    suffix_sum_logprob = per_pos_logprob.sum(dim=1)
    suffix_avg_logprob = per_pos_logprob.mean(dim=1)

    pred_next = suffix_logits.argmax(dim=-1)
    exact_hit_per_sample = (pred_next == suffix_targets).float().mean(dim=1)

    k = int(min(topk, suffix_logits.size(-1)))
    topk_ids = torch.topk(suffix_logits, k=k, dim=-1).indices
    topk_hit_per_sample = (topk_ids == suffix_targets.unsqueeze(-1)).any(dim=-1).float().mean(dim=1)

    per_pos_topk_hit = (topk_ids == suffix_targets.unsqueeze(-1)).any(dim=-1).float()  # [B,S]

    target_logits = torch.gather(suffix_logits, -1, suffix_targets.unsqueeze(-1))
    rank = (suffix_logits > target_logits).sum(dim=-1) + 1  # [B,S], 1=best
    avg_rank_per_sample = rank.float().mean(dim=1)

    return {
        'suffix_sum_logprob': float(suffix_sum_logprob.mean().item()),
        'suffix_avg_logprob': float(suffix_avg_logprob.mean().item()),
        'per_position_logprob_curve': per_pos_logprob.mean(dim=0).detach().cpu(),
        'exact_hit': float(exact_hit_per_sample.mean().item()),
        'topk_hit': float(topk_hit_per_sample.mean().item()),
        'per_position_topk_hit_curve': per_pos_topk_hit.mean(dim=0).detach().cpu(),
        'avg_rank': float(avg_rank_per_sample.mean().item()),
        'rank_tensor': rank.detach().cpu(),
    }


def run_condition_for_model(model_key: str, current_input_ids: torch.Tensor, target_full_tokens: torch.Tensor):
    model = MODELS[model_key]
    with torch.no_grad():
        outputs = model(current_input_ids, return_cache=True, return_intermediates=True)
    metrics = suffix_metrics_from_logits(
        logits=outputs['logits'],
        target_full_tokens=target_full_tokens,
        prefix_len=prefix_len,
        topk=5,
    )
    return outputs, metrics


def print_metrics(title: str, model_key: str, metrics: dict):
    prefix = f'[{model_key}][{title}]'
    print(f"{prefix} suffix_sum_logprob = {metrics['suffix_sum_logprob']:.6f}")
    print(f"{prefix} suffix_avg_logprob = {metrics['suffix_avg_logprob']:.6f}")
    print(f"{prefix} exact_hit = {metrics['exact_hit']:.4f}")
    print(f"{prefix} topk_hit = {metrics['topk_hit']:.4f}")
    print(f"{prefix} avg_rank = {metrics['avg_rank']:.3f}")


def metric_delta(left_metrics: dict, right_metrics: dict):
    return {
        'suffix_avg_logprob': float(left_metrics['suffix_avg_logprob'] - right_metrics['suffix_avg_logprob']),
        'suffix_sum_logprob': float(left_metrics['suffix_sum_logprob'] - right_metrics['suffix_sum_logprob']),
        'exact_hit': float(left_metrics['exact_hit'] - right_metrics['exact_hit']),
        'topk_hit': float(left_metrics['topk_hit'] - right_metrics['topk_hit']),
        'avg_rank': float(left_metrics['avg_rank'] - right_metrics['avg_rank']),
    }


def build_delta_report(clean_metrics: dict, corrupted_metrics: dict, patched_metrics: dict):
    clean_minus_corrupted = metric_delta(clean_metrics, corrupted_metrics)
    patched_minus_corrupted = metric_delta(patched_metrics, corrupted_metrics)
    patched_minus_clean = metric_delta(patched_metrics, clean_metrics)

    if abs(clean_minus_corrupted['suffix_avg_logprob']) > 1e-8:
        recovery_ratio = patched_minus_corrupted['suffix_avg_logprob'] / clean_minus_corrupted['suffix_avg_logprob']
    else:
        recovery_ratio = float('nan')

    return {
        'clean_minus_corrupted': clean_minus_corrupted,
        'patched_minus_corrupted': patched_minus_corrupted,
        'patched_minus_clean': patched_minus_clean,
        'recovery_ratio': recovery_ratio,
    }


In [ ]:
# Cell 6. clean / corrupted 基线比较（双模型同样本、同扰动对比）
clean_input_ids = full_clean[:, :-1]
model_results = {}
candidate_scoreboard = {}

for model_key in MODEL_ORDER:
    clean_outputs, clean_metrics = run_condition_for_model(model_key, clean_input_ids, target_full_tokens=full_clean)
    print_metrics('clean', model_key, clean_metrics)

    candidate_results = {}
    for name, full_cor in corruption_candidates.items():
        cor_input_ids = full_cor[:, :-1]
        _, cor_metrics = run_condition_for_model(model_key, cor_input_ids, target_full_tokens=full_clean)
        corrupted_drop = clean_metrics['suffix_avg_logprob'] - cor_metrics['suffix_avg_logprob']
        candidate_results[name] = {
            'metrics': cor_metrics,
            'corrupted_drop': float(corrupted_drop),
        }
        candidate_scoreboard.setdefault(name, {'drops_by_model': {}})
        candidate_scoreboard[name]['drops_by_model'][model_key] = float(corrupted_drop)

    model_results[model_key] = {
        'clean_outputs': clean_outputs,
        'clean_metrics': clean_metrics,
        'candidate_results': candidate_results,
    }

for name, payload in candidate_scoreboard.items():
    drops = list(payload['drops_by_model'].values())
    payload['mean_corrupted_drop'] = float(sum(drops) / len(drops))
    payload['min_corrupted_drop'] = float(min(drops))

selected_corruption_name = max(
    candidate_scoreboard.keys(),
    key=lambda name: (
        candidate_scoreboard[name]['mean_corrupted_drop'],
        candidate_scoreboard[name]['min_corrupted_drop'],
    ),
)
corrupted_full = corruption_candidates[selected_corruption_name]
corrupted_input_ids = corrupted_full[:, :-1]

print('\ncorruption candidates by aggregated drop (mean over models):')
for name, payload in sorted(
    candidate_scoreboard.items(),
    key=lambda kv: (kv[1]['mean_corrupted_drop'], kv[1]['min_corrupted_drop']),
    reverse=True,
):
    print(
        f"- {name}: mean_drop={payload['mean_corrupted_drop']:+.6f}, "
        f"min_drop={payload['min_corrupted_drop']:+.6f}, drops={payload['drops_by_model']}"
    )

print(f"\nselected_corruption(shared across models): {selected_corruption_name}")
print('clean prefix sample0:', full_clean[0, :prefix_len].tolist())
print('corrupted prefix sample0:', corrupted_full[0, :prefix_len].tolist())
print('changed positions sample0:', (corrupted_full[0, :prefix_len] != full_clean[0, :prefix_len]).nonzero(as_tuple=False).view(-1).tolist())

for model_key in MODEL_ORDER:
    selected_metrics = model_results[model_key]['candidate_results'][selected_corruption_name]['metrics']
    selected_drop = model_results[model_key]['candidate_results'][selected_corruption_name]['corrupted_drop']
    model_results[model_key]['corrupted_metrics'] = selected_metrics
    model_results[model_key]['selected_corrupted_drop'] = float(selected_drop)
    print_metrics('corrupted(selected)', model_key, selected_metrics)
    print(f"[{model_key}][corrupted(selected)] drop = {selected_drop:+.6f}")
    if selected_drop < 0.01:
        print(f"[WARNING][{model_key}] 当前样本/扰动仍偏弱（drop < 0.01），后续 patch 信号可能仍有限。")

baseline_selection_summary = {
    'selected_corruption_name': selected_corruption_name,
    'selected_corruption_example': {
        'clean_prefix_sample0': full_clean[0, :prefix_len].tolist(),
        'corrupted_prefix_sample0': corrupted_full[0, :prefix_len].tolist(),
        'changed_positions_sample0': (corrupted_full[0, :prefix_len] != full_clean[0, :prefix_len]).nonzero(as_tuple=False).view(-1).tolist(),
    },
    'candidate_rankings': [
        {
            'name': name,
            'mean_corrupted_drop': payload['mean_corrupted_drop'],
            'min_corrupted_drop': payload['min_corrupted_drop'],
            'drops_by_model': payload['drops_by_model'],
        }
        for name, payload in sorted(
            candidate_scoreboard.items(),
            key=lambda kv: (kv[1]['mean_corrupted_drop'], kv[1]['min_corrupted_drop']),
            reverse=True,
        )
    ],
    'model_results': {
        model_key: {
            'clean_metrics': scalar_metrics_view(model_results[model_key]['clean_metrics']),
            'corrupted_metrics': scalar_metrics_view(model_results[model_key]['corrupted_metrics']),
            'selected_corrupted_drop': model_results[model_key]['selected_corrupted_drop'],
        }
        for model_key in MODEL_ORDER
    },
}
save_json_artifact(baseline_selection_summary, 'baseline_selection_summary.json')


In [ ]:
# Cell 7. patched 条件比较（双模型各自 site sweep，再汇总对比）
preferred_sites = [
    'blocks.0.attn_out',
    'blocks.0.output',
    'blocks.1.attn_out',
    'blocks.1.output',
    'embedding_out',
]

for model_key in MODEL_ORDER:
    clean_outputs = model_results[model_key]['clean_outputs']
    clean_metrics = model_results[model_key]['clean_metrics']
    corrupted_metrics = model_results[model_key]['corrupted_metrics']

    clean_cache = clean_outputs['cache']
    cache_keys = sorted(clean_cache.data.keys())
    available_sites = [s for s in preferred_sites if s in clean_cache.data]
    if not available_sites:
        # 回退：从 cache 中选若干稳定位点
        available_sites = [k for k in cache_keys if k.endswith(('attn_out', 'mlp_out', 'output'))][:6]
    if not available_sites:
        raise RuntimeError(f'[{model_key}] 未找到可 patch 的位点。')

    site_results = {}
    for site in available_sites:
        overrides = patch_from_cache(clean_cache, site)
        with torch.no_grad():
            patched_outputs = MODELS[model_key](
                corrupted_input_ids,
                return_cache=True,
                return_intermediates=True,
                activation_overrides=overrides,
            )
        patched_metrics = suffix_metrics_from_logits(
            logits=patched_outputs['logits'],
            target_full_tokens=full_clean,
            prefix_len=prefix_len,
            topk=5,
        )
        recovery_gain = patched_metrics['suffix_avg_logprob'] - corrupted_metrics['suffix_avg_logprob']
        site_results[site] = {
            'patched_outputs': patched_outputs,
            'patched_metrics': patched_metrics,
            'recovery_gain': float(recovery_gain),
        }

    selected_site = max(site_results.keys(), key=lambda site: site_results[site]['recovery_gain'])
    patched_outputs = site_results[selected_site]['patched_outputs']
    patched_metrics = site_results[selected_site]['patched_metrics']
    delta_report = build_delta_report(clean_metrics, corrupted_metrics, patched_metrics)

    model_results[model_key]['available_sites'] = available_sites
    model_results[model_key]['site_results'] = site_results
    model_results[model_key]['selected_patch_site'] = selected_site
    model_results[model_key]['patched_outputs'] = patched_outputs
    model_results[model_key]['patched_metrics'] = patched_metrics
    model_results[model_key]['delta_report'] = delta_report

    print(f"\n[{model_key}] patch site recovery_gain ranking:")
    for site, payload in sorted(site_results.items(), key=lambda kv: kv[1]['recovery_gain'], reverse=True):
        print(f"- {site:24s} recovery_gain={payload['recovery_gain']:+.6f}")

    print(f"\n[{model_key}] selected_patch_site: {selected_site}")
    print_metrics('patched(selected_site)', model_key, patched_metrics)
    print(json.dumps(delta_report, indent=2, ensure_ascii=False))

patch_recovery_summary = {
    'selected_corruption_name': selected_corruption_name,
    'preferred_sites': preferred_sites,
    'model_patch_results': {
        model_key: {
            'selected_patch_site': model_results[model_key]['selected_patch_site'],
            'available_sites': model_results[model_key]['available_sites'],
            'site_rankings': [
                {
                    'site': site,
                    'recovery_gain': payload['recovery_gain'],
                }
                for site, payload in sorted(
                    model_results[model_key]['site_results'].items(),
                    key=lambda kv: kv[1]['recovery_gain'],
                    reverse=True,
                )
            ],
            'clean_metrics': scalar_metrics_view(model_results[model_key]['clean_metrics']),
            'corrupted_metrics': scalar_metrics_view(model_results[model_key]['corrupted_metrics']),
            'patched_metrics': scalar_metrics_view(model_results[model_key]['patched_metrics']),
            'delta_report': model_results[model_key]['delta_report'],
        }
        for model_key in MODEL_ORDER
    },
}
save_json_artifact(patch_recovery_summary, 'patch_recovery_summary.json')


In [ ]:
# Cell 8. 可视化（双模型对比）
condition_order = ['clean', 'corrupted', 'patched']
condition_metric_lookup = {
    condition: f'{condition}_metrics'
    for condition in condition_order
}


def grouped_metric_bar(axis, metric_key: str, title: str, ylabel: str, *, ylim=None, show_legend: bool = False):
    x = list(range(len(condition_order)))
    width = 0.8 / max(1, len(MODEL_ORDER))
    offsets = [
        (idx - (len(MODEL_ORDER) - 1) / 2) * width
        for idx in range(len(MODEL_ORDER))
    ]
    for idx, model_key in enumerate(MODEL_ORDER):
        values = [model_results[model_key][condition_metric_lookup[condition]][metric_key] for condition in condition_order]
        axis.bar([i + offsets[idx] for i in x], values, width=width, label=model_key)
    axis.set_xticks(x)
    axis.set_xticklabels(condition_order)
    axis.set_title(title)
    axis.set_ylabel(ylabel)
    if ylim is not None:
        axis.set_ylim(*ylim)
    if show_legend:
        axis.legend()


fig, axes = plt.subplots(2, 4, figsize=(24, 10))

grouped_metric_bar(axes[0, 0], 'suffix_avg_logprob', 'Suffix Avg Logprob', 'logprob', show_legend=True)
grouped_metric_bar(axes[0, 1], 'suffix_sum_logprob', 'Suffix Sum Logprob', 'logprob')
grouped_metric_bar(axes[0, 2], 'exact_hit', 'Exact Hit', 'ratio', ylim=(0.0, 1.0))
grouped_metric_bar(axes[0, 3], 'topk_hit', 'Top-k Hit', 'ratio', ylim=(0.0, 1.0))
grouped_metric_bar(axes[1, 0], 'avg_rank', 'Average Target Token Rank (lower is better)', 'rank')

# 6) drop / recovery（按模型并列）
delta_labels = ['clean-corrupted', 'patched-corrupted', 'patched-clean']
delta_keys = ['clean_minus_corrupted', 'patched_minus_corrupted', 'patched_minus_clean']
delta_x = list(range(len(delta_labels)))
delta_width = 0.8 / max(1, len(MODEL_ORDER))
delta_offsets = [
    (idx - (len(MODEL_ORDER) - 1) / 2) * delta_width
    for idx in range(len(MODEL_ORDER))
]
for idx, model_key in enumerate(MODEL_ORDER):
    values = [model_results[model_key]['delta_report'][key]['suffix_avg_logprob'] for key in delta_keys]
    axes[1, 1].bar([i + delta_offsets[idx] for i in delta_x], values, width=delta_width, label=model_key)
axes[1, 1].set_xticks(delta_x)
axes[1, 1].set_xticklabels(delta_labels, rotation=15)
axes[1, 1].axhline(0.0, color='black', linewidth=1)
axes[1, 1].set_title('Drop / Recovery Gain (suffix avg logprob)')
axes[1, 1].legend()

# 7) per-position suffix token logprob curve
line_styles = {
    'clean': '-',
    'corrupted': '--',
    'patched': ':',
}
for model_key in MODEL_ORDER:
    for condition in condition_order:
        metrics = model_results[model_key][condition_metric_lookup[condition]]
        x = list(range(len(metrics['per_position_logprob_curve'])))
        axes[1, 2].plot(
            x,
            metrics['per_position_logprob_curve'].tolist(),
            marker='o',
            linestyle=line_styles[condition],
            label=f'{model_key}-{condition}',
        )
axes[1, 2].set_title('Per-position Suffix Token Logprob')
axes[1, 2].set_xlabel('suffix position')
axes[1, 2].set_ylabel('logprob')
axes[1, 2].legend(fontsize=8)

# 8) per-position top-k hit curve
for model_key in MODEL_ORDER:
    for condition in condition_order:
        metrics = model_results[model_key][condition_metric_lookup[condition]]
        x = list(range(len(metrics['per_position_topk_hit_curve'])))
        axes[1, 3].plot(
            x,
            metrics['per_position_topk_hit_curve'].tolist(),
            marker='o',
            linestyle=line_styles[condition],
            label=f'{model_key}-{condition}',
        )
axes[1, 3].set_title('Per-position Top-k Hit')
axes[1, 3].set_xlabel('suffix position')
axes[1, 3].set_ylabel('hit ratio')
axes[1, 3].set_ylim(0.0, 1.0)
axes[1, 3].legend(fontsize=8)

plt.tight_layout()
save_figure_artifact('prefix_suffix_scoring_model_comparison.png', fig)
plt.show()

summary_table = {
    model_key: [
        {
            'condition': condition,
            **scalar_metrics_view(model_results[model_key][condition_metric_lookup[condition]]),
        }
        for condition in condition_order
    ]
    for model_key in MODEL_ORDER
}
print('\ncondition summary by model:')
print(json.dumps(summary_table, indent=2, ensure_ascii=False))

delta_table = {
    model_key: model_results[model_key]['delta_report']
    for model_key in MODEL_ORDER
}
print('\ndelta summary by model:')
print(json.dumps(delta_table, indent=2, ensure_ascii=False))

comparison_table = [
    {
        'model_key': model_key,
        'selected_corruption_drop': model_results[model_key]['selected_corrupted_drop'],
        'selected_patch_site': model_results[model_key]['selected_patch_site'],
        'recovery_ratio': model_results[model_key]['delta_report']['recovery_ratio'],
    }
    for model_key in MODEL_ORDER
]
print('\ncomparison summary:')
print(json.dumps(comparison_table, indent=2, ensure_ascii=False))

save_json_artifact(
    {
        'selected_corruption_name': selected_corruption_name,
        'summary_table': summary_table,
        'delta_table': delta_table,
        'comparison_table': comparison_table,
    },
    'visualization_summary.json',
)


## Cell 9. 结果解释

本 revised notebook 现在改成双模型共同加载 / 对比，增强点是：
1. `standard` 与 `attnres` 同时加载，并固定在同一批样本上评分；
2. corruption 阶段不再只服务单模型，而是先比较每个候选在两个模型上的 drop，再选一个共享扰动做对比；
3. patched 阶段对每个模型分别做 site sweep，最后统一比较 recovery gain、最佳 patch site 和 recovery ratio；
4. 可视化改成双模型 grouped bars + per-position curves，便于直接横向对照。

这仍然是 Prefix-conditioned target suffix 的 teacher-forced scoring：
- 评分始终基于 logits + 固定 suffix target；
- 没有做 generate/sample，也不是完整 extraction attack。

如果这版仍然信号弱，最可能原因通常是：
- 当前 checkpoint 对该样本簇本身不确定；
- corruption 仍未命中真正决定 suffix 的条件位；
- patch site sweep 的候选范围仍然不够贴近真正起作用的表示位点。
